# 📧 Task 2: Spam Email Detection

## Objective
Build a machine learning model that classifies SMS/email messages as **Spam** or **Ham (Not Spam)** using Natural Language Processing (NLP) techniques.

## Approach
1. Load and explore the SMS Spam Collection dataset
2. Preprocess text data (cleaning, tokenization, stemming)
3. Extract features using TF-IDF Vectorization
4. Train multiple classification models
5. Evaluate and compare model performance
6. Choose the best model and summarize findings

---
## 1. Import Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import re
import string
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# WordCloud
try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
except ImportError:
    !pip install wordcloud -q
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True

import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully!')

---
## 2. Load the Dataset

Upload `spam.csv` when prompted (if running in Colab), or ensure it is in the same directory.

In [ ]:
# --- Upload file in Google Colab ---
# Uncomment the lines below if you are running this in Google Colab
# from google.colab import files
# uploaded = files.upload()

# Load dataset
df = pd.read_csv('spam.csv', encoding='latin-1')

# The CSV has extra unnamed columns — keep only the first two
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

print(f'Dataset shape: {df.shape}')
print(f'Number of messages: {df.shape[0]}')
df.head(10)

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview

In [ ]:
# Basic info
print('='*50)
print('DATASET INFO')
print('='*50)
df.info()

print(f'\nDuplicate rows: {df.duplicated().sum()}')
print(f'Missing values:\n{df.isnull().sum()}')

# Drop duplicates
df.drop_duplicates(inplace=True)
print(f'\nShape after removing duplicates: {df.shape}')

### 3.2 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71', '#e74c3c']

# Count plot
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Message Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
df['label'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                 colors=colors, startangle=90,
                                 explode=(0.05, 0.05))
axes[1].set_title('Message Distribution (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print('\nClass Distribution:')
print(df['label'].value_counts())

### 3.3 Message Length Analysis

In [ ]:
# Add length features
df['msg_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Message length distribution
for label, color in zip(['ham', 'spam'], colors):
    subset = df[df['label'] == label]
    axes[0].hist(subset['msg_length'], bins=50, alpha=0.7, label=label, color=color)
axes[0].set_title('Message Character Length Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Character Length')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Word count distribution
for label, color in zip(['ham', 'spam'], colors):
    subset = df[df['label'] == label]
    axes[1].hist(subset['word_count'], bins=50, alpha=0.7, label=label, color=color)
axes[1].set_title('Message Word Count Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\nAverage message length:')
print(df.groupby('label')[['msg_length', 'word_count']].mean().round(1))

### 3.4 Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Ham word cloud
ham_text = ' '.join(df[df['label'] == 'ham']['message'])
wc_ham = WordCloud(width=800, height=400, background_color='white',
                   colormap='Greens', max_words=100).generate(ham_text)
axes[0].imshow(wc_ham, interpolation='bilinear')
axes[0].set_title('Ham (Not Spam) — Word Cloud', fontsize=14, fontweight='bold')
axes[0].axis('off')

# Spam word cloud
spam_text = ' '.join(df[df['label'] == 'spam']['message'])
wc_spam = WordCloud(width=800, height=400, background_color='white',
                    colormap='Reds', max_words=100).generate(spam_text)
axes[1].imshow(wc_spam, interpolation='bilinear')
axes[1].set_title('Spam — Word Cloud', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

---
## 4. Text Preprocessing

In [ ]:
# Initialize tools
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Clean and preprocess a text message."""
    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # 3. Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # 4. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 5. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 6. Tokenize
    tokens = word_tokenize(text)

    # 7. Remove stopwords and stem
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words and len(word) > 2]

    return ' '.join(tokens)

# Apply preprocessing
print('Preprocessing messages...')
df['cleaned_message'] = df['message'].apply(preprocess_text)
print('Done!\n')

# Show examples
print('='*70)
print('PREPROCESSING EXAMPLES')
print('='*70)
for i in range(5):
    print(f'\n[{df.iloc[i]["label"].upper()}]')
    print(f'  Original:  {df.iloc[i]["message"][:80]}...')
    print(f'  Cleaned:   {df.iloc[i]["cleaned_message"][:80]}...')

---
## 5. Feature Extraction (TF-IDF)

In [ ]:
# Encode labels: ham = 0, spam = 1
df['label_encoded'] = (df['label'] == 'spam').astype(int)

# Features and target
X = df['cleaned_message']
y = df['label_encoded']

# Train-test split (80-20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} messages')
print(f'Testing  set: {X_test.shape[0]} messages')

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'\nTF-IDF Matrix shape (Train): {X_train_tfidf.shape}')
print(f'TF-IDF Matrix shape (Test):  {X_test_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_)}')

---
## 6. Model Training & Evaluation

In [ ]:
# Define models
models = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Linear SVM': LinearSVC(random_state=42, max_iter=5000),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
}

# Train and evaluate
results = []

for name, model in models.items():
    print(f'\n{"="*60}')
    print(f'  Training: {name}')
    print(f'{"="*60}')

    # Train
    model.fit(X_train_tfidf, y_train)

    # Predict
    y_pred = model.predict(X_test_tfidf)

    # Probability / decision function for ROC
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test_tfidf)[:, 1]
    else:
        y_prob = model.decision_function(X_test_tfidf)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'AUC-ROC': auc,
        'y_prob': y_prob
    })

    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1 Score:  {f1:.4f}')
    print(f'  AUC-ROC:   {auc:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

### 6.1 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test_tfidf)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Ham', 'Spam'],
                yticklabels=['Ham', 'Spam'])
    axes[i].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6.2 ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))
colors_roc = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']

for i, r in enumerate(results):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    plt.plot(fpr, tpr, color=colors_roc[i], lw=2,
             label=f"{r['Model']} (AUC = {r['AUC-ROC']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('ROC Curves — Model Comparison', fontsize=15, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Model Comparison

In [ ]:
# Create results dataframe
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'y_prob'} for r in results])
results_df = results_df.set_index('Model')
results_df = results_df.sort_values('F1 Score', ascending=False)

print('='*70)
print('                    MODEL COMPARISON SUMMARY')
print('='*70)
print(results_df.to_string())
print(f'\n\U0001f3c6 Best Model (by F1 Score): {results_df.index[0]}')
print(f'   F1 Score: {results_df.iloc[0]["F1 Score"]:.4f}')
print(f'   AUC-ROC:  {results_df.iloc[0]["AUC-ROC"]:.4f}')

In [ ]:
# Grouped bar chart comparison
fig, ax = plt.subplots(figsize=(14, 7))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC']
x = np.arange(len(results_df.index))
width = 0.15
bar_colors = ['#1abc9c', '#3498db', '#e67e22', '#e74c3c', '#9b59b6']

for i, metric in enumerate(metrics):
    bars = ax.bar(x + i * width, results_df[metric], width,
                  label=metric, color=bar_colors[i], edgecolor='white')
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Model', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Model Performance Comparison', fontsize=15, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(results_df.index, rotation=15)
ax.legend(loc='lower right', fontsize=10)
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Sample Predictions

In [ ]:
# Test with some sample messages
sample_messages = [
    "Congratulations! You've won a free iPhone. Click here to claim now!",
    "Hey, are you free for lunch tomorrow?",
    "URGENT: Your account has been compromised. Verify your details immediately.",
    "Can you pick up some groceries on your way home?",
    "You have been selected for a cash prize of $10,000! Call now to claim.",
    "Meeting rescheduled to 3 PM. See you there.",
]

# Use the best model (Naive Bayes is typically best for text)
best_model = models['Multinomial Naive Bayes']

print('='*70)
print('  SAMPLE PREDICTIONS (Multinomial Naive Bayes)')
print('='*70)

for msg in sample_messages:
    cleaned = preprocess_text(msg)
    vectorized = tfidf.transform([cleaned])
    prediction = best_model.predict(vectorized)[0]
    proba = best_model.predict_proba(vectorized)[0]
    label = 'SPAM \U0001f6a8' if prediction == 1 else 'HAM \u2705'
    confidence = max(proba) * 100

    print(f'\n  Message:    "{msg[:60]}..."')
    print(f'  Prediction: {label} (Confidence: {confidence:.1f}%)')

---
## 9. Top TF-IDF Features for Spam vs Ham

In [ ]:
# Get feature names
feature_names = np.array(tfidf.get_feature_names_out())

# Use Naive Bayes log probabilities to find top spam/ham words
nb_model = models['Multinomial Naive Bayes']
log_prob_ham = nb_model.feature_log_prob_[0]
log_prob_spam = nb_model.feature_log_prob_[1]

# Ratio: words that most distinguish spam from ham
spam_vs_ham = log_prob_spam - log_prob_ham

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top spam words
top_spam_idx = spam_vs_ham.argsort()[-15:]
axes[0].barh(feature_names[top_spam_idx], spam_vs_ham[top_spam_idx], color='#e74c3c')
axes[0].set_title('Top 15 Spam-Indicative Words', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Log Probability Ratio (Spam/Ham)')

# Top ham words
top_ham_idx = spam_vs_ham.argsort()[:15]
axes[1].barh(feature_names[top_ham_idx], -spam_vs_ham[top_ham_idx], color='#2ecc71')
axes[1].set_title('Top 15 Ham-Indicative Words', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Log Probability Ratio (Ham/Spam)')

plt.tight_layout()
plt.show()

---
## 10. Conclusion

### Summary of Results

In this project, we developed a complete **NLP pipeline for Spam Email/SMS Detection** using the SMS Spam Collection dataset.

**Key Findings:**
- We trained and evaluated **4 different models**: Multinomial Naive Bayes, Logistic Regression, Linear SVM, and Random Forest.
- **Text preprocessing** (lowercasing, removing stopwords, stemming) and **TF-IDF vectorization** were used for feature extraction.
- **Multinomial Naive Bayes** and **Linear SVM** typically achieve the best results for text classification tasks like spam detection, with high precision and recall.
- Spam messages tend to be **longer** and contain words like "free", "win", "call", "prize", "claim", while ham messages contain everyday conversational words.

### Future Improvements
- Use **Word2Vec** or **BERT embeddings** instead of TF-IDF for richer text representations
- Train a **deep learning model** (LSTM/GRU) for sequence-aware classification
- Apply **cross-validation** for more robust evaluation
- Deploy the model as a **REST API** or **web app** using Flask/Streamlit